In [ ]:
import time
import json
import re
import threading
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

def setup_driver(headless=False):
    options = webdriver.ChromeOptions()
    options.binary_location = r"C:\Users\ABHISHEK\AppData\Local\BraveSoftware\Brave-Browser\Application\brave.exe"
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920x1080")
    options.add_argument("--disable-extensions")
    options.add_argument("--log-level=3")
    if headless:
        options.add_argument("--headless")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def extract_playlists(driver):
    driver.get("https://www.youtube.com/music")
    driver.maximize_window()
    WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    while True:
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(1)
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

    soup = BeautifulSoup(driver.page_source, "html.parser")
    all_playlists = {}
    sections = soup.find_all("h2")

    for section in sections:
        section_name = section.text.strip()
        section_playlists = []
        next_element = section.find_next()
        while next_element and next_element.name != "h2":
            for a_tag in next_element.find_all("a", href=True):
                href = a_tag["href"]
                if "/playlist" in href:
                    full_link = "https://www.youtube.com" + href
                    cleaned_link = re.sub(r"&playnext=1&index=\d+", "", full_link)
                    playlist_name_tag = a_tag.find_next("span")
                    if playlist_name_tag:
                        playlist_name = playlist_name_tag.text.strip()
                        if playlist_name and not any(p["url"] == cleaned_link for p in section_playlists) and len(section_playlists) < 10:
                            # Extract playlist image (thumbnail)
                            img_tag = a_tag.find_next("img")
                            thumbnail_url = img_tag["src"] if img_tag and img_tag.get("src") else "No Thumbnail"
                            
                            section_playlists.append({
                                "name": playlist_name,
                                "url": cleaned_link,
                                "thumbnail_url": thumbnail_url
                            })
            next_element = next_element.find_next()
        if section_playlists:
            all_playlists[section_name] = section_playlists[:10]  # Ensure only 10 playlists per section
    return all_playlists

def get_playlist_videos(driver, playlist_url, limit=5):
    driver.get(playlist_url)
    time.sleep(2)
    for _ in range(3):
        driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
        time.sleep(1)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    videos = []
    for video in soup.select("a#video-title")[:limit]:
        title = video.get("title", "No Title")
        video_url = "https://www.youtube.com" + video["href"]
        img_tag = video.find_parent("ytd-playlist-video-renderer").select_one("img")
        thumbnail_url = img_tag["src"] if img_tag and img_tag.get("src") else "No Thumbnail"
        videos.append({"title": title, "video_url": video_url, "thumbnail_url": thumbnail_url})
    return videos

def process_playlist(bucket_name, playlists, results):
    driver = setup_driver(headless=True)
    results[bucket_name] = []
    for playlist in playlists:
        videos = get_playlist_videos(driver, playlist["url"])
        results[bucket_name].append({"playlist_name": playlist["name"], "videos": videos})
    driver.quit()

def main():
    driver = setup_driver()
    playlists_data = extract_playlists(driver)
    driver.quit()
    
    # Save extracted playlists data to a file
    with open("youtube_music_playlists.json", "w", encoding="utf-8") as f:
        json.dump(playlists_data, f, indent=2, ensure_ascii=False)
    
    all_results = {}
    threads = []
    for bucket_name, playlists in playlists_data.items():
        thread = threading.Thread(target=process_playlist, args=(bucket_name, playlists, all_results))
        threads.append(thread)
        thread.start()
    
    for thread in threads:
        thread.join()
    
    # Save final results (videos data) to a file
    with open("data.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=4, ensure_ascii=False)
    
    print("✅ Extraction complete. Data saved to data.json")

if __name__ == "__main__":
    main()